# Stage 1 — zero-shot difficulty calibration (Colab)

**This notebook runs calibration only.** It does not run seed selection,
strategy induction, H0, Active, Passive, hidden-test evaluation or token
matching, and it must not be extended to do so.

It answers one question: for each problem family, which difficulty tier gives
meaningful but nontrivial zero-shot solving accuracy?

Order:
1. install requirements
2. print GPU name and VRAM
3. run the non-GPU mathematical self-tests — **stop here if they fail**
4. run a tiny calibration smoke test (`configs/calibration_smoke.yaml`)
5. display the calibration summary
6. stop

The smoke test is 4-bit and uses 2 problems per tier. **Its accuracies must not
be used to select a difficulty tier** — that requires the lab-GPU run with
`configs/calibration.yaml`.

## 1. Get the code and install requirements

In [ ]:
import os, sys
REPO = "/content/active-falsification"
if not os.path.exists(REPO):
    # replace with your clone URL, or upload the directory to /content
    !git clone https://example.invalid/active-falsification.git {REPO} || echo "clone the repo or upload it to /content/active-falsification"
os.chdir(REPO)
print(os.getcwd())

In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes

## 2. GPU name and VRAM

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/2**30:.1f} GiB free / {total/2**30:.1f} GiB total")
else:
    print("No GPU. Runtime > Change runtime type > GPU (T4 is enough for the 4-bit smoke test).")

## 3. Mathematical self-tests — mandatory

Do not run a model until this prints `SELF TEST PASSED`.

In [ ]:
!python scripts/self_test.py

## 4. The exact zero-shot prompt

Read this before running anything. It must be a general solver instruction plus
the problem text, with no worked examples, no strategy description, no
applicability information and nothing about difficulty.

In [ ]:
import sys
sys.path.insert(0, "src")
import prompts
print("SYSTEM:")
print(prompts.SOLVE_SYSTEM)
print()
print("USER:", repr(prompts.SOLVE_USER))
print()
print("prompt hash:", prompts.prompts_hash())

### Preview the calibration problem bank

Same bank for every model, generated once from the master seed.

In [ ]:
from calibration import build_problem_bank
from families import all_families

fams = all_families()
built = build_problem_bank(fams, 5, 20260901)
print("validation failures:", built["validation_failures"])
for name in fams:
    for tier in ("easy", "medium", "hard"):
        items = built["bank"][name][tier]
        print(f"\n--- {name} / {tier} ---")
        for p in items[:2]:
            print(f"  {p.text}")
            print(f"    exact answer: {p.exact_answer}")

## 5. Offline plumbing check (no GPU, seconds)

In [ ]:
!python scripts/run_calibration.py --config configs/calibration_mock.yaml --skip-self-test

## 6. Calibration smoke test on the GPU

One small model, 4-bit, 2 problems per tier = 18 generations. This only checks
that the code executes on a GPU.

In [ ]:
!python scripts/run_calibration.py --config configs/calibration_smoke.yaml --tag smoke

## 7. Display the calibration summary

In [ ]:
import glob, os, json
import pandas as pd

RUN = sorted(glob.glob("outputs/*_calibration_smoke_smoke"))[-1]
print("run:", RUN)

print("\n--- per model x family x difficulty ---")
display(pd.read_csv(os.path.join(RUN, "calibration_summary.csv")))

print("\n--- pooled over models ---")
display(pd.read_csv(os.path.join(RUN, "pooled_difficulty_summary.csv")))

print("\n--- selected difficulties ---")
print(json.dumps(json.load(open(os.path.join(RUN, "selected_difficulties.json"))), indent=2)[:4000])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(RUN, "calibration_report.md")).read()))

### Read some raw responses

Never trust an accuracy you have not traced back to raw output.

In [ ]:
rows = [json.loads(l) for l in open(os.path.join(RUN, "calibration_runs.jsonl"))]
for r in rows[:3]:
    print("=" * 72)
    print(r["family"], "|", r["difficulty"], "| gold:", r["gold_answer"],
          "| parsed:", r["parsed_answer"], "| correct:", r["correct"],
          "| unparsed:", r["unparsed"])
    print(r["problem_text"])
    print("-" * 72)
    print(r["raw_response"][:1200])

## 8. Stop here

This notebook ends at Stage 1.

For the real calibration, run on the lab GPU:

```bash
python scripts/run_calibration.py --config configs/calibration.yaml
```

and return the whole `outputs/<RUN_ID>/` directory to the authors.

Do not run `run_pilot.py`, `run_final.py`, or anything involving seeds,
induction or adaptation until the authors have approved the selected tiers.